In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
%run /Workspace/Users/gayatrijoshi663@gmail.com/regis-healthcare/1_setup/utility

In [0]:
# %run /Workspace/Users/dhotepatil00@gmail.com/regis-healthcare/1_setup/utility

In [0]:
print(bronze_schema,silver_schema,gold_schema) 

In [0]:
dbutils.widgets.text("catalog","regis_healthcare","catalog")
dbutils.widgets.text("data_source","medications","data_source")

In [0]:
catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")

#### Silver Processing

In [0]:
df_bronze = spark.sql(f"select * from {catalog}.{bronze_schema}.{data_source};")
display(df_bronze)
print(df_bronze.count())

In [0]:
# schema check
print(df_bronze.count())
df_bronze.printSchema()

In [0]:
df_bronze.columns

In [0]:
# drop duplicate
df_silver = df_bronze.dropDuplicates()
print(df_silver.count())

In [0]:
df_silver = df_silver.withColumn(
    "medication_id",
    F.trim(F.col("medication_id"))
).withColumn(
    "resident_id",
    F.trim(F.col("resident_id"))
).withColumn(
    "medication_name",
    F.trim(F.col("medication_name"))
).withColumn(
    "dosage",
    F.trim(F.col("dosage"))
).withColumn(
    "frequency",
    F.trim(F.col("frequency"))
).withColumn(
    "prescribing_doctor",
    F.trim(F.col("prescribing_doctor"))
).withColumn(
    "start_date",
    F.trim(F.col("start_date"))
).withColumn(
    "end_date",
    F.trim(F.col("end_date"))
).withColumn(
    "status",
    F.trim(F.col("status"))
).withColumn(
    "notes",
    F.trim(F.col("notes"))
).withColumn(
    "created_at",
    F.trim(F.col("created_at"))
)

In [0]:
# null records count 
from pyspark.sql.functions import col,count,when
null_count = df_silver.select([count(when(col(c).isNull(),c)).alias(c)for c in df_silver.columns
                               ])
display(null_count)

#### Cleaning data in table

In [0]:
# medication_id',
check = df_silver.filter(~col("medication_id").rlike("^MED"))
display(check)

df_silver = df_silver.withColumn("medication_id",when(~col("medication_id").rlike("^MED"),None).otherwise(col("medication_id")))
display(df_silver)


In [0]:
#  'resident_id',
check = df_silver.filter(~col("resident_id").rlike("^RES"))
display(check)

df_silver = df_silver.withColumn("resident_id",when(~col("resident_id").rlike("^RES"),None).otherwise(col("resident_id")))
display(df_silver)



In [0]:
#  'medication_name',
from pyspark.sql.functions import col,when,trim,initcap

df_silver = df_silver.withColumn("medication_name",initcap(trim(col("medication_name"))))

dup = df_silver.groupBy("medication_name").count().filter(col("count")>1)
dup.display()


In [0]:
#  'dosage'


In [0]:
# medication_id',
#  'resident_id',
#  'medication_name',
#  'dosage',
#  'frequency',
#  'prescribing_doctor',
#  'start_date',
#  'end_date',
#  'status',
#  'notes',
#  'created_at',